In [0]:
# After Silver and Gold tables are created,this Notebook checks whether the data is accurate, complete, consistent, and safe to use for analytics and ready for pipeline creation.

# WHAT WE ARE DOING:
# 1. Load required Silver and Gold tables
# 2. Validate row counts
# 3. Validate primary keys
# 4. Validate foreign keys
# 5. Check critical null values
# 6. Validate dates and quantities
# 7. Check negative transaction amounts
# 8. Reconcile revenue and delivered units
# 9. Generate the final data-quality status
#INPUT:
# Silver and Gold Delta tables as input

# OUTPUT:
# Data-quality validation results


In [0]:
# Load Required Tables
# Purpose:
# Load the Silver and Gold tables that we want to validate.
# This notebook does not transform data.
# It only checks whether the pipeline produced valid data.

from pyspark.sql import functions as F

# Database/schema where our project tables are stored
catalog_schema = "workspace.indian_ecommerce_sales_analytics"

# -----------------------------
# Load Silver tables
# -----------------------------

silver_customers = spark.table(
    f"{catalog_schema}.silver_customers"
)

silver_products = spark.table(
    f"{catalog_schema}.silver_products"
)

silver_sales = spark.table(
    f"{catalog_schema}.silver_sales"
)

# -----------------------------
# Load Gold dimension tables
# -----------------------------

dim_customer = spark.table(
    f"{catalog_schema}.dim_customer"
)

dim_product = spark.table(
    f"{catalog_schema}.dim_product"
)

dim_date = spark.table(
    f"{catalog_schema}.dim_date"
)

# -----------------------------
# Load Gold fact table
# -----------------------------

fact_sales = spark.table(
    f"{catalog_schema}.fact_sales"
)

print("All required tables loaded successfully.")

All required tables loaded successfully.


In [0]:
# Row Count Checks
# Purpose:
# Verify that the expected number of records are present
# after the Bronze -> Silver -> Gold transformations.

# Expected record counts based on the source dataset
expected_counts = {
    "silver_customers": 40000,
    "silver_products": 2000,
    "silver_sales": 250000,
    "dim_customer": 40000,
    "dim_product": 2000,
    "fact_sales": 250000
}

# Get actual row counts from the tables
actual_counts = {
    "silver_customers": silver_customers.count(),
    "silver_products": silver_products.count(),
    "silver_sales": silver_sales.count(),
    "dim_customer": dim_customer.count(),
    "dim_product": dim_product.count(),
    "fact_sales": fact_sales.count()
}

# Store validation results
row_count_results = []

# Compare expected count with actual count
for table_name, expected_count in expected_counts.items():

    actual_count = actual_counts[table_name]

    # PASS when expected and actual counts are equal
    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    row_count_results.append(
        (
            table_name,
            expected_count,
            actual_count,
            status
        )
    )

# Convert the results into a Spark DataFrame
row_count_df = spark.createDataFrame(
    row_count_results,
    [
        "Table_Name",
        "Expected_Count",
        "Actual_Count",
        "Status"
    ]
)

display(row_count_df)

Table_Name,Expected_Count,Actual_Count,Status
silver_customers,40000,40000,PASS
silver_products,2000,2000,PASS
silver_sales,250000,250000,PASS
dim_customer,40000,40000,PASS
dim_product,2000,2000,PASS
fact_sales,250000,250000,PASS


In [0]:
# Primary Key Checks
# Purpose:
# Verify that primary key columns contain unique values.

# Customer_ID -> Customer table
# Product_ID  -> Product table
# Order_ID    -> Sales table

# -----------------------------
# Customer primary key
# -----------------------------

customer_total = silver_customers.count()

customer_distinct = (
    silver_customers
    .select("Customer_ID")
    .distinct()
    .count()
)

customer_pk_status = (
    "PASS"
    if customer_total == customer_distinct
    else "FAIL"
)

# -----------------------------
# Product primary key
# -----------------------------

product_total = silver_products.count()

product_distinct = (
    silver_products
    .select("Product_ID")
    .distinct()
    .count()
)

product_pk_status = (
    "PASS"
    if product_total == product_distinct
    else "FAIL"
)

# -----------------------------
# Order primary key
# -----------------------------

sales_total = silver_sales.count()

sales_distinct = (
    silver_sales
    .select("Order_ID")
    .distinct()
    .count()
)

order_pk_status = (
    "PASS"
    if sales_total == sales_distinct
    else "FAIL"
)

# Display results
print(f"Customer PK: {customer_pk_status}")
print(f"Customer Rows: {customer_total}")
print(f"Distinct Customer_IDs: {customer_distinct}")

print()

print(f"Product PK: {product_pk_status}")
print(f"Product Rows: {product_total}")
print(f"Distinct Product_IDs: {product_distinct}")

print()

print(f"Order PK: {order_pk_status}")
print(f"Order Rows: {sales_total}")
print(f"Distinct Order_IDs: {sales_distinct}")

Customer PK: PASS
Customer Rows: 40000
Distinct Customer_IDs: 40000

Product PK: PASS
Product Rows: 2000
Distinct Product_IDs: 2000

Order PK: PASS
Order Rows: 250000
Distinct Order_IDs: 250000


In [0]:
# Foreign Key Checks
# Purpose:
# Verify that every Customer_ID and Product_ID in the fact table
# exists in the corresponding dimension table.

# ------------------------------------------------------------
# Check Customer_ID relationship
# fact_sales.Customer_ID -> dim_customer.Customer_ID
# ------------------------------------------------------------

invalid_customer_fk = (
    fact_sales
    .join(
        dim_customer.select("Customer_ID"),
        on="Customer_ID",
        how="left_anti"
    )
    .count()
)

customer_fk_status = (
    "PASS"
    if invalid_customer_fk == 0
    else "FAIL"
)

# ------------------------------------------------------------
# Check Product_ID relationship
# fact_sales.Product_ID -> dim_product.Product_ID
# ------------------------------------------------------------

invalid_product_fk = (
    fact_sales
    .join(
        dim_product.select("Product_ID"),
        on="Product_ID",
        how="left_anti"
    )
    .count()
)

product_fk_status = (
    "PASS"
    if invalid_product_fk == 0
    else "FAIL"
)

# Display results
print(f"Invalid Customer Foreign Keys: {invalid_customer_fk}")
print(f"Customer FK Status: {customer_fk_status}")

print()

print(f"Invalid Product Foreign Keys: {invalid_product_fk}")
print(f"Product FK Status: {product_fk_status}")

Invalid Customer Foreign Keys: 0
Customer FK Status: PASS

Invalid Product Foreign Keys: 0
Product FK Status: PASS


In [0]:
# Critical Null Checks

# Purpose:
# Check important columns in the fact table for NULL values.
#
# These columns are critical because they are required for
# joins, calculations and business analysis.

critical_columns = [
    "Customer_ID",
    "Product_ID",
    "Order_ID",
    "Order_Date",
    "Quantity",
    "Realized_Revenue"
]

null_results = []

# Check each important column
for column_name in critical_columns:

    # Count NULL values in the current column
    null_count = (
        fact_sales
        .filter(
            F.col(column_name).isNull()
        )
        .count()
    )

    # PASS when no NULL values are found
    status = (
        "PASS"
        if null_count == 0
        else "FAIL"
    )

    null_results.append(
        (
            column_name,
            null_count,
            status
        )
    )

# Create a validation report
null_check_df = spark.createDataFrame(
    null_results,
    [
        "Column_Name",
        "Null_Count",
        "Status"
    ]
)

display(null_check_df)

Column_Name,Null_Count,Status
Customer_ID,0,PASS
Product_ID,0,PASS
Order_ID,0,PASS
Order_Date,0,PASS
Quantity,0,PASS
Realized_Revenue,0,PASS


In [0]:
# ============================================
# Cell 4 - Foreign Key Checks
# ============================================

# Customer FK
invalid_customer_fk = (
    fact_sales
    .join(
        dim_customer.select("Customer_ID"),
        on="Customer_ID",
        how="left_anti"
    )
    .count()
)

customer_fk_status = (
    "PASS"
    if invalid_customer_fk == 0
    else "FAIL"
)


# Product FK
invalid_product_fk = (
    fact_sales
    .join(
        dim_product.select("Product_ID"),
        on="Product_ID",
        how="left_anti"
    )
    .count()
)

product_fk_status = (
    "PASS"
    if invalid_product_fk == 0
    else "FAIL"
)


# Date FK
invalid_date_fk = (
    fact_sales
    .join(
        dim_date.select("Date_Key"),
        on="Date_Key",
        how="left_anti"
    )
    .count()
)

date_fk_status = (
    "PASS"
    if invalid_date_fk == 0
    else "FAIL"
)


print(f"Invalid Customer FK: {invalid_customer_fk}")
print(f"Customer FK Status: {customer_fk_status}")
print()

print(f"Invalid Product FK: {invalid_product_fk}")
print(f"Product FK Status: {product_fk_status}")
print()

print(f"Invalid Date FK: {invalid_date_fk}")
print(f"Date FK Status: {date_fk_status}")

Invalid Customer FK: 0
Customer FK Status: PASS

Invalid Product FK: 0
Product FK Status: PASS

Invalid Date FK: 0
Date FK Status: PASS


In [0]:
# Business Rule Checks
# Purpose:
# Validate whether important business rules are satisfied.

# Rules:
# 1. Delivery date should not be before order date.
# 2. Quantity should be greater than zero.
# 3. Order status should not be NULL or blank.

# ------------------------------------------------------------
# Rule 1: Delivery Date Validation
# ------------------------------------------------------------
# Delivery_Date cannot be earlier than Order_Date.

invalid_delivery_dates = (
    silver_sales
    .filter(
        F.col("Delivery_Date") < F.col("Order_Date")
    )
    .count()
)

date_status = (
    "PASS"
    if invalid_delivery_dates == 0
    else "FAIL"
)

# ------------------------------------------------------------
# Rule 2: Quantity Validation
# ------------------------------------------------------------
# Quantity should always be greater than zero.

invalid_quantity = (
    silver_sales
    .filter(
        F.col("Quantity") <= 0
    )
    .count()
)

quantity_status = (
    "PASS"
    if invalid_quantity == 0
    else "FAIL"
)

# ------------------------------------------------------------
# Rule 3: Order Status Validation
# ------------------------------------------------------------
# Check for NULL or blank status values.
# We do not hardcode the valid status list here because
# this dataset may contain additional legitimate statuses.

invalid_order_status = (
    silver_sales
    .filter(
        F.col("Order_Status").isNull()
        |
        (F.trim(F.col("Order_Status")) == "")
    )
    .count()
)

order_status_validation = (
    "PASS"
    if invalid_order_status == 0
    else "FAIL"
)

# Display results
print(f"Invalid Delivery Dates: {invalid_delivery_dates}")
print(f"Delivery Date Status: {date_status}")

print()

print(f"Invalid Quantity Records: {invalid_quantity}")
print(f"Quantity Status: {quantity_status}")

print()

print(f"NULL/Blank Order Status Records: {invalid_order_status}")
print(f"Order Status Validation: {order_status_validation}")

Invalid Delivery Dates: 0
Delivery Date Status: PASS

Invalid Quantity Records: 0
Quantity Status: PASS

NULL/Blank Order Status Records: 0
Order Status Validation: PASS


In [0]:
# Cell 7 - Gold KPI Reconciliation

# Purpose:
# Verify that the Gold analytics KPIs match the values
# calculated directly from the Gold fact table.

# This confirms that our business calculations are correct.

# Load Gold total sales table
gold_total_sales = spark.table(
    f"{catalog_schema}.gold_total_sales"
)

# ------------------------------------------------------------
# Revenue Reconciliation
# ------------------------------------------------------------

# Calculate realized revenue directly from fact_sales
fact_revenue = (
    fact_sales
    .agg(
        F.sum("Realized_Revenue").alias("Revenue")
    )
    .collect()[0]["Revenue"]
)

# Read revenue from Gold KPI table
gold_revenue = (
    gold_total_sales
    .select("Total_Gross_Revenue")
    .collect()[0]["Total_Gross_Revenue"]
)

# Calculate difference
revenue_difference = abs(
    float(fact_revenue) -
    float(gold_revenue)
)

# Revenue passes when difference is less than 1 paisa
revenue_status = (
    "PASS"
    if revenue_difference < 0.01
    else "FAIL"
)

# ------------------------------------------------------------
# Units Reconciliation
# ------------------------------------------------------------

# Only Delivered orders represent realized units sold.
delivered_fact_units = (
    fact_sales
    .filter(
        F.col("Is_Delivered") == True
    )
    .agg(
        F.sum("Quantity").alias("Units")
    )
    .collect()[0]["Units"]
)

# Read delivered units from Gold KPI table
gold_units = (
    gold_total_sales
    .select("Total_Units_Sold")
    .collect()[0]["Total_Units_Sold"]
)

# Calculate difference
units_difference = abs(
    int(delivered_fact_units) -
    int(gold_units)
)

units_status = (
    "PASS"
    if units_difference == 0
    else "FAIL"
)

# Display reconciliation results
print("REVENUE RECONCILIATION")
print("-" * 40)
print(f"Fact Revenue: ₹{fact_revenue:,.2f}")
print(f"Gold Revenue: ₹{gold_revenue:,.2f}")
print(f"Difference: ₹{revenue_difference:,.2f}")
print(f"Status: {revenue_status}")

print()

print("UNITS RECONCILIATION")
print("-" * 40)
print(f"Delivered Fact Units: {delivered_fact_units:,}")
print(f"Gold Units: {gold_units:,}")
print(f"Difference: {units_difference:,}")
print(f"Status: {units_status}")

REVENUE RECONCILIATION
----------------------------------------
Fact Revenue: ₹4,741,566,354.41
Gold Revenue: ₹4,741,566,354.41
Difference: ₹0.00
Status: PASS

UNITS RECONCILIATION
----------------------------------------
Delivered Fact Units: 249,834
Gold Units: 249,834
Difference: 0
Status: PASS


In [0]:
# Cell 8 - Final Data Quality Summary
# Purpose:
# Combine all individual validation results into one final
# data quality report.
#
# PASS    -> Check succeeded
# WARNING -> Data issue exists but pipeline can continue
# FAIL    -> Critical data quality issue

quality_results = [

    # -----------------------------
    # Row count checks
    # -----------------------------

    (
        "Row Count - Customers",
        "PASS"
        if actual_counts["silver_customers"] == 40000
        else "FAIL"
    ),

    (
        "Row Count - Products",
        "PASS"
        if actual_counts["silver_products"] == 2000
        else "FAIL"
    ),

    (
        "Row Count - Sales",
        "PASS"
        if actual_counts["silver_sales"] == 250000
        else "FAIL"
    ),

    # -----------------------------
    # Primary key checks
    # -----------------------------

    ("Customer Primary Key", customer_pk_status),
    ("Product Primary Key", product_pk_status),
    ("Order Primary Key", order_pk_status),

    # -----------------------------
    # Foreign key checks
    # -----------------------------

    ("Customer Foreign Key", customer_fk_status),
    ("Product Foreign Key", product_fk_status),

    # -----------------------------
    # Null checks
    # -----------------------------

    (
        "Critical Null Checks",
        "PASS"
        if all(
            row[2] == "PASS"
            for row in null_results
        )
        else "FAIL"
    ),

    # -----------------------------
    # Business rule checks
    # -----------------------------

    ("Delivery Date Validation", date_status),
    ("Quantity Validation", quantity_status),
    ("Order Status Validation", order_status_validation),

    # -----------------------------
    # Reconciliation checks
    # -----------------------------

    ("Gold Revenue Reconciliation", revenue_status),
    ("Gold Units Reconciliation", units_status)
]

# Convert all results into a Spark DataFrame
quality_summary = spark.createDataFrame(
    quality_results,
    [
        "Quality_Check",
        "Status"
    ]
)

display(quality_summary)

Quality_Check,Status
Row Count - Customers,PASS
Row Count - Products,PASS
Row Count - Sales,PASS
Customer Primary Key,PASS
Product Primary Key,PASS
Order Primary Key,PASS
Customer Foreign Key,PASS
Product Foreign Key,PASS
Critical Null Checks,PASS
Delivery Date Validation,PASS


In [0]:
# Overall Quality Status
# Purpose:
# Calculate the final status of the complete data quality check.
#
# FAIL     -> At least one critical check failed.
# WARNING  -> Some non-critical issue needs attention.
# PASS     -> No critical checks failed.

# Count failed checks
failed_checks = (
    quality_summary
    .filter(
        F.col("Status") == "FAIL"
    )
    .count()
)

# Count warning checks
warning_checks = (
    quality_summary
    .filter(
        F.col("Status") == "WARNING"
    )
    .count()
)

# Determine overall status
if failed_checks == 0:
    overall_status = "PASS"
else:
    overall_status = "FAIL"

# Print final result
print("=" * 50)
print("FINAL DATA QUALITY RESULT")
print("=" * 50)

print(f"Failed Checks   : {failed_checks}")
print(f"Warning Checks  : {warning_checks}")
print(f"Overall Status  : {overall_status}")

print("=" * 50)

FINAL DATA QUALITY RESULT
Failed Checks   : 0
Warning Checks  : 0
Overall Status  : PASS
